In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_excel("Cohort_readydataset.xlsx")

In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,CohortMonth,InvoiceMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-01-12 08:26:00,2.55,17850,United Kingdom,15.30,2010-01,2010-01
1,536365,71053,WHITE METAL LANTERN,6,2010-01-12 08:26:00,3.39,17850,United Kingdom,20.34,2010-01,2010-01
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-01-12 08:26:00,2.75,17850,United Kingdom,22.00,2010-01,2010-01
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-01-12 08:26:00,3.39,17850,United Kingdom,20.34,2010-01,2010-01
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-01-12 08:26:00,3.39,17850,United Kingdom,20.34,2010-01,2010-01


# Cohort Index Creation

In [4]:
df['InvoiceMonth'] = pd.to_datetime(df['InvoiceMonth'].astype(str))
df['CohortMonth'] = pd.to_datetime(df['CohortMonth'].astype(str))

In [5]:
invoice_year = df['InvoiceMonth'].dt.year
invoice_month = df['InvoiceMonth'].dt.month

cohort_year = df['CohortMonth'].dt.year
cohort_month = df['CohortMonth'].dt.month

df['CohortIndex'] = (
    (invoice_year - cohort_year) * 12
    + (invoice_month - cohort_month)
    + 1
)

In [6]:
df[['CohortMonth','InvoiceMonth','CohortIndex']].head()

,CohortMonth,InvoiceMonth,CohortIndex
0,2010-01-01,2010-01-01,1
1,2010-01-01,2010-01-01,1
2,2010-01-01,2010-01-01,1
3,2010-01-01,2010-01-01,1
4,2010-01-01,2010-01-01,1


# Retention Matrix Creation

In [7]:
cohort_data = df.groupby(
    ['CohortMonth','CohortIndex']
)['CustomerID'].nunique().reset_index()

In [8]:
retention_matrix = cohort_data.pivot(
    index='CohortMonth',
    columns='CohortIndex',
    values='CustomerID'
)

In [9]:
retention_matrix

CohortIndex,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
CohortMonth,,,,,,,,,,,,,,,,,,,,,
2010-01-01,95.0,6.0,4.0,NaN,5.0,7.0,3.0,10.0,7.0,5.0,...,31.0,34.0,34.0,34.0,34.0,20.0,7.0,17.0,19.0,15.0
2010-02-01,93.0,NaN,NaN,NaN,2.0,6.0,3.0,7.0,4.0,NaN,...,28.0,30.0,30.0,31.0,9.0,14.0,13.0,16.0,10.0,NaN
2010-03-01,46.0,NaN,1.0,1.0,1.0,NaN,3.0,NaN,NaN,12.0,...,18.0,12.0,18.0,7.0,5.0,10.0,6.0,6.0,NaN,NaN
2010-05-01,69.0,2.0,3.0,1.0,4.0,1.0,NaN,25.0,34.0,18.0,...,30.0,14.0,15.0,12.0,10.0,15.0,NaN,NaN,NaN,NaN
2010-06-01,70.0,2.0,2.0,1.0,1.0,NaN,21.0,22.0,18.0,19.0,...,7.0,10.0,4.0,8.0,2.0,NaN,NaN,NaN,NaN,NaN
2010-07-01,50.0,NaN,1.0,1.0,NaN,6.0,17.0,17.0,18.0,17.0,...,10.0,5.0,6.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN
2010-08-01,83.0,NaN,1.0,NaN,20.0,31.0,21.0,32.0,26.0,31.0,...,12.0,13.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-09-01,67.0,NaN,NaN,15.0,24.0,15.0,24.0,21.0,22.0,17.0,...,5.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-10-01,40.0,NaN,9.0,12.0,13.0,15.0,15.0,9.0,9.0,11.0,...,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Retention Rate Analysis

In [10]:
retention_rate = retention_matrix.divide(
    retention_matrix.iloc[:,0],
    axis=0
)

In [11]:
retention_rate.round(3)

CohortIndex,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
CohortMonth,,,,,,,,,,,,,,,,,,,,,
2010-01-01,1.0,0.063,0.042,NaN,0.053,0.074,0.032,0.105,0.074,0.053,...,0.326,0.358,0.358,0.358,0.358,0.211,0.074,0.179,0.200,0.158
2010-02-01,1.0,NaN,NaN,NaN,0.022,0.065,0.032,0.075,0.043,NaN,...,0.301,0.323,0.323,0.333,0.097,0.151,0.140,0.172,0.108,NaN
2010-03-01,1.0,NaN,0.022,0.022,0.022,NaN,0.065,NaN,NaN,0.261,...,0.391,0.261,0.391,0.152,0.109,0.217,0.130,0.130,NaN,NaN
2010-05-01,1.0,0.029,0.043,0.014,0.058,0.014,NaN,0.362,0.493,0.261,...,0.435,0.203,0.217,0.174,0.145,0.217,NaN,NaN,NaN,NaN
2010-06-01,1.0,0.029,0.029,0.014,0.014,NaN,0.300,0.314,0.257,0.271,...,0.100,0.143,0.057,0.114,0.029,NaN,NaN,NaN,NaN,NaN
2010-07-01,1.0,NaN,0.020,0.020,NaN,0.120,0.340,0.340,0.360,0.340,...,0.200,0.100,0.120,0.080,NaN,NaN,NaN,NaN,NaN,NaN
2010-08-01,1.0,NaN,0.012,NaN,0.241,0.373,0.253,0.386,0.313,0.373,...,0.145,0.157,0.145,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-09-01,1.0,NaN,NaN,0.224,0.358,0.224,0.358,0.313,0.328,0.254,...,0.075,0.119,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-10-01,1.0,NaN,0.225,0.300,0.325,0.375,0.375,0.225,0.225,0.275,...,0.075,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
